In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# 1. Load the secret API key from your .env file
load_dotenv()

# 2. Define the exact shape of the data we want (The "Bouncer")
# This guarantees the AI gives us exactly these fields and data types
class TransactionData(BaseModel):
    merchant_name: str = Field(description="The clean name of the shop, person, or business. E.g., 'Raju Tea Stall'")
    amount: float = Field(description="The exact amount of money spent as a number")
    payment_method: str = Field(description="How it was paid (e.g., UPI, Card, Netbanking)")
    raw_upi_id: str = Field(description="The raw VPA/UPI ID if present (e.g., 'xyz@paytm'), otherwise 'N/A'")

# 3. Wake up the AI (Using your brand new gemini-2.5-flash!)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0  # 0 means zero creativity, just stick strictly to the facts
)

# 4. Force the AI to use our strict Pydantic rules
structured_llm = llm.with_structured_output(TransactionData)

# 5. The messy SMS (The Test)
messy_sms = "Rs 240.00 debited from a/c **1234 on 05-03-26 to VPA-paytmqr2810050501@paytm (Raju Tea Stall). Ref: 4321"

print("🧠 Sending SMS to Gemini 2.5 Flash...\n")

# 6. Run the AI!
result = structured_llm.invoke(messy_sms)

# 7. Print the beautifully structured data
print("--- 🎯 AI EXTRACTION RESULT ---")
print(f"Merchant: {result.merchant_name}")
print(f"Amount:   ₹{result.amount}")
print(f"Method:   {result.payment_method}")
print(f"UPI ID:   {result.raw_upi_id}")

print("\n--- 💻 RAW JSON EQUIVALENT ---")
print(result.model_dump_json(indent=2))

c:\Users\JAYARAM\Desktop\SpendSense-AI\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


🧠 Sending SMS to Gemini 2.5 Flash...

--- 🎯 AI EXTRACTION RESULT ---
Merchant: Raju Tea Stall
Amount:   ₹240.0
Method:   UPI
UPI ID:   paytmqr2810050501@paytm

--- 💻 RAW JSON EQUIVALENT ---
{
  "merchant_name": "Raju Tea Stall",
  "amount": 240.0,
  "payment_method": "UPI",
  "raw_upi_id": "paytmqr2810050501@paytm"
}
